In [0]:
# MAGIC %run ../bronze/bronze_dq_checks

In [0]:
from pyspark.sql import DataFrame, window, functions as F


In [0]:
date_formats = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy", "yyyy/MM/dd", "MMM d, yyyy"]

PARSED_DATE = "coalesce("+", ".join(
    f"try_to_timestamp(trim(order_date), '{f}')" for f in date_formats
)+ ")"


NULL_TOKENS = ("", "NULL", "N/A", "NA", "NONE", "NAN")
EMAIL_REGEX = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+[.][A-Za-z]{2,}$'

COUNTRY_MAP = {
    "in": "India", "india": "India",
    "us": "United States", "usa": "United States", "united states": "United States", "united states of america": "United States",
    "uk": "UNited Kingdom", "united kingdom": "United Kingdom",
    "uae": "United Arab Emirates"
}


PAYMENT_MAP = {
    "credit card": "Credit Card",
    "debit card": "Debit Card",
    "upi": "UPI",
    "cod": "Cash on Delivery", "cash on delivery": "Cash on Delivery",
    "net banking": "Net Banking",
    "paypal": "PayPal"
}


VALID_STATUSES = ("Delivered", "Shipped", "Processing", "Cancelled", "Returned")

NON_REVENUE_STATUSES = {"Cancelled", "Returned"}


In [0]:
def blank_to_null(c):
    col = F.trim(F.col(c) if isinstance(c, str) else c)
    return F.when(F.upper(col).isin(*NULL_TOKENS), F.lit(None).otherwise(col))


def title_case(col):
    return F.initcap(F.regexp_replace(col, r"\s+", " "))

def map_values(col, mapping: dict, default):
    expr = None
    for key, value in mapping.items():
        expr = F.when(col == key, F.lit(value)) if expr is None else expr.when(col== key, F.lit(value))

    return expr.otherwise(default)

def normalize_phone(raw):
    digits = F.regexp_replace(blank_to_null(raw), r"[^0-9]", "")
    return (
        F.when(digits.rlike("^0*$"), F.lit(None))
        .when(F.length(digits) < 7, F.lit(None))
        .when(F.length(digits == 12) & digits.startswith("91"), F.substring(digits, 3, 10))
        .otherwise(digits)
    )



In [0]:
def clean_columns(df: DataFrame):
    name = title_case(blank_to_null("customer_name"))
    email_raw = F.lower(blank_to_null("customer_email"))
    email = F.when(email_raw.rlike(EMAIL_REGEX), email_raw)
    phone =  normalize_phone("phone_number")
    city = title_case(blank_to_null("city"))

    country_raw = blank_to_null("country")
    country = map_values(F.lower(country_raw), COUNTRY_MAP, title_case(country_raw))

    payment_raw = blank_to_null("payment_method")
    payment_key = F.lower(F.regexp_replace(payment_raw, r"[_\s]+", " "))
    payment = map_values(payment_key, PAYMENT_MAP, F.initcap(payment_key))

    status_raw = F.initcap(blank_to_null("order_status"))
    status = F.when(status_raw.isin(*VALID_STATUSES), status_raw).otherwise(F.lit("unknown"))

    discount_raw = F.expr("try_cast(discount_percent AS DECIMAL(5, 2))")
    discount = F.when(discount_raw.between(0, 100), discount_raw).otherwise(F.lit(0).cast("decimal(5, 2)"))

    